# 03 – The Librarian: Advanced Hybrid RAG

**Operation Ledger-Mind: The Financial Intelligence**

In [1]:
# Setup
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
from typing import List, Dict, Any

sys.path.append(str(Path.cwd().parent))
load_dotenv()

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings
)

config = load_config("../src/config/config.yaml")
llm = get_llm(config)
embeddings = get_text_embeddings(config)
print("✅ LLM and embeddings ready")

/Users/sasiriakalanka/Large Documents/Programmes/AI/Zuu/AI/operation_ledger_mind/src/services/llm_services.py:129: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


✅ LLM and embeddings ready


In [2]:
# Load and chunk PDF (reuse from Notebook 01)
import fitz
import re

def clean_text(text):
    return re.sub(r'\s+', ' ', re.sub(r'\x00', '', text)).strip()

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = "".join([page.get_text() for page in doc])
    doc.close()
    return clean_text(text)

def chunk_text(text, chunk_size=1500, overlap=150):
    """Split text into chunks - FIXED VERSION."""
    chunks = []
    chunk_id = 0

    # Use simple stepping (safer)
    step = chunk_size - overlap

    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))

        # Try sentence break
        if end < len(text):
            for p in ['.', '!', '?']:
                pos = text.rfind(p, end-200, end)
                if pos > start:
                    end = pos + 1
                    break

        chunk = text[start:end].strip()
        if chunk:
            chunks.append({
                'chunk_id': chunk_id,
                'text': chunk,
                'start': start,
                'end': end
            })
            chunk_id += 1

        if end >= len(text):
            break

    return chunks

# Test it
pdf_path = Path(config['data_root']) / "2024-Annual-Report.pdf"
full_text = extract_text_from_pdf(str(pdf_path))
print(f"Text length: {len(full_text):,} characters")

chunks = chunk_text(full_text, 1500, 150)
print(f"✅ Created {len(chunks)} chunks")
print(f"   Average length: {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")

Text length: 620,206 characters
✅ Created 460 chunks
   Average length: 1430 chars


In [3]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

client = weaviate.connect_to_embedded(
    version="1.27.3"
)

collection_name = "UberFinancials"

if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

collection = client.collections.create(
    name=collection_name,
    vector_config=Configure.Vectors.self_provided(),
    properties=[
        Property(name="chunk_id", data_type=DataType.INT),
        Property(name="text", data_type=DataType.TEXT),
        Property(name="start_char", data_type=DataType.INT),
        Property(name="end_char", data_type=DataType.INT)
    ]
)

print(f"✅ Created collection: {collection_name}")

{"action":"startup","build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-05-09T07:56:35+05:30"}
{"action":"startup","auto_schema_enabled":true,"build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"auto schema enabled setting is set to \"true\"","time":"2026-05-09T07:56:35+05:30"}
{"build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"No resource limits set, weaviate will use all available memory and CPU. To limit resources, set LIMIT_RESOURCES=true","time":"2026-05-09T07:56:35+05:30"}
{"build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","

✅ Created collection: UberFinancials


In [4]:
# Index chunks
chunk_texts = [c['text'] for c in chunks]
vectors = embeddings.embed_documents(chunk_texts)

collection = client.collections.get(collection_name)

with collection.batch.dynamic() as batch:
    for chunk, vector in zip(chunks, vectors):
        batch.add_object(
            properties={
                "chunk_id": chunk['chunk_id'],
                "text": chunk['text'],
                "start_char": chunk['start'],
                "end_char": chunk['end']
            },
            vector=vector
        )

collection.aggregate.over_all(total_count=True)
print(f"✅ Indexed {len(chunks)} chunks")

{"action":"telemetry_push","build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"telemetry started","payload":"\u0026{MachineID:7483ffc0-8841-4cbd-a094-dfede7397110 Type:INIT Version:1.27.3 NumObjects:0 OS:darwin Arch:arm64 UsedModules:[]}","time":"2026-05-09T07:56:39+05:30"}


✅ Indexed 460 chunks


In [5]:
# Search functions
def dense_search(query, top_k=10):
    query_vector = embeddings.embed_query(query)
    collection = client.collections.get(collection_name)
    response = collection.query.near_vector(near_vector=query_vector, limit=top_k, return_metadata=['distance'])
    return [{'chunk_id': obj.properties['chunk_id'], 'text': obj.properties['text'], 'score': 1 - obj.metadata.distance} 
            for obj in response.objects]

def bm25_search(query, top_k=10):
    collection = client.collections.get(collection_name)
    response = collection.query.bm25(query=query, limit=top_k)
    return [{'chunk_id': obj.properties['chunk_id'], 'text': obj.properties['text'], 'score': obj.metadata.score or 0.0}
            for obj in response.objects]

print("✅ Search functions ready")

✅ Search functions ready


In [6]:
# Reciprocal Rank Fusion
def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    rrf_scores = {}
    
    for rank, result in enumerate(dense_results, 1):
        cid = result['chunk_id']
        if cid not in rrf_scores:
            rrf_scores[cid] = {'text': result['text'], 'chunk_id': cid, 'rrf_score': 0.0}
        rrf_scores[cid]['rrf_score'] += 1 / (k + rank)
    
    for rank, result in enumerate(bm25_results, 1):
        cid = result['chunk_id']
        if cid not in rrf_scores:
            rrf_scores[cid] = {'text': result['text'], 'chunk_id': cid, 'rrf_score': 0.0}
        rrf_scores[cid]['rrf_score'] += 1 / (k + rank)
    
    return sorted(rrf_scores.values(), key=lambda x: x['rrf_score'], reverse=True)

print("✅ RRF ready")

✅ RRF ready


In [7]:
# Cross-encoder reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=5):
    pairs = [[query, c['text']] for c in candidates]
    scores = reranker.predict(pairs)
    
    for candidate, score in zip(candidates, scores):
        candidate['rerank_score'] = float(score)
    
    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:top_k]

print("✅ Reranker ready")

✅ Reranker ready


In [8]:
# Complete RAG pipeline
def query_librarian(question, return_sources=False):
    # Hybrid retrieval
    dense = dense_search(question, top_k=10)
    bm25 = bm25_search(question, top_k=10)
    
    # Fusion
    fused = reciprocal_rank_fusion(dense, bm25)
    
    # Rerank
    reranked = rerank(question, fused, top_k=5)
    
    # Build context
    context = "\n\n---\n\n".join([r['text'] for r in reranked])
    
    # Generate
    prompt = f'''Answer based ONLY on context from Uber's 2024 Annual Report.

    Context:
    {context}

    Question: {question}

    Answer:'''
    
    response = llm.invoke(prompt)
    answer = response.content
    
    if return_sources:
        return {'answer': answer, 'sources': reranked, 'context': context}
    return answer

print("✅ query_librarian() function ready")
print("\n🧪 Testing...")
print(query_librarian("What is Uber's mission?"))
print("\n✅ The Librarian is ready!")
client.close()

✅ query_librarian() function ready

🧪 Testing...
Uber's mission is to "reimagine the way the world moves for the better."

✅ The Librarian is ready!


{"action":"restapi_management","build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"Shutting down... ","time":"2026-05-09T07:56:51+05:30","version":"1.27.3"}
{"action":"restapi_management","build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"Stopped serving weaviate at http://127.0.0.1:8079","time":"2026-05-09T07:56:51+05:30","version":"1.27.3"}
{"action":"telemetry_push","build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3","level":"info","msg":"telemetry terminated","payload":"\u0026{MachineID:7483ffc0-8841-4cbd-a094-dfede7397110 Type:TERMINATE Version:1.27.3 NumObjects:0 OS:darwin Arch:arm64 UsedModules:[none]}","time":"2026-05-09T07:56:52+05:30"}
{"build_git_commit":"4258bdfc2","build_go_version":"go1.23.3","build_image_tag":"HEAD","build_wv_version":"1.27.3",